In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect('games.duckdb')

In [5]:
genres = con.execute('SELECT * FROM genres').df()

In [7]:
franchises = con.execute('SELECT * FROM franchises').df()

In [8]:
games = con.execute('SELECT * FROM games').df()

In [23]:
games_processed = games.copy()
games_processed['main_genre_id'] = games_processed['genres'].str[0]

exploded = franchises.explode('games')

merged_games = exploded.merge(
    games_processed,
    left_on='games',
    right_on='id',
    suffixes=('_franchise', '_game')
)

full_data = merged_games.merge(
    genres,
    left_on='main_genre_id',
    right_on='id'
)

genre_counts = full_data.groupby(['id_franchise', 'name']).size().reset_index(name='count')
top_genres = genre_counts.sort_values('count', ascending=False) \
                         .groupby('id_franchise') \
                         .head(1) \
                         .rename(columns={'name': 'franchise_genre'})

franchises = franchises.merge(
    top_genres[['id_franchise', 'franchise_genre']],
    left_on='id',
    right_on='id_franchise',
    how='left'
)

if 'id_franchise' in franchises.columns:
    franchises = franchises.drop(columns=['id_franchise'])

In [24]:
franchises

,id,name,games,franchise_lifespan,franchise_duration,franchise_genre_x,franchise_genre_y,franchise_genre
0,4289,Space Dandy,[62563],0,0.00,NaN,NaN,NaN
1,6766,Lizards Must Die,<NA>,<NA>,NaN,NaN,NaN,NaN
2,6458,Masyanya,[335429],0,0.00,Point-and-click,Point-and-click,Point-and-click
3,295,The Dog,[4815],0,0.00,Role-playing (RPG),Role-playing (RPG),Role-playing (RPG)
4,556,The Bard's Tale,"[3141, 3803, 26986, 2431, 2430, 2429, 36893, 3...",33,32.71,Role-playing (RPG),Role-playing (RPG),Role-playing (RPG)
...,...,...,...,...,...,...,...,...
2487,1651,Bemani,"[80106, 112148, 127142, 79244, 126031, 128003,...",27,27.27,Music,Music,Music
2488,7038,Asteria,<NA>,<NA>,NaN,NaN,NaN,NaN
2489,7055,Wheel of Time,<NA>,<NA>,NaN,NaN,NaN,NaN
2490,7058,Sukisyo!,<NA>,<NA>,NaN,NaN,NaN,NaN


In [25]:
con.close()